# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "group 24"
student_name = "Shameel Zeshan Khader Sheriff"
student_id = "26030371"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# No aditional package required

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [ ]:
# addition package to import

import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
use_case_definition = """
Business use case:
The business use case is product portfolio segmentation for margin-aware promotion decisions. The retailer wants to identify actionable groups of sold products based on demand, margin position, and promotion behaviour so that the merchandising team can decide which products should be protected from unnecessary discounting, supported with targeted offers, or reviewed because promotion activity may be weakening commercial value.

Data mining problem:
This is a clustering problem because there is no predefined product segment label. The aim is to group sold products with similar commercial behaviour using product-level sales demand, sales value, price, cost, margin, promotion eligibility, and realised discount behaviour.

Business decision supported:
The model supports promotion and product portfolio review. Each cluster will help the business identify product roles such as high-margin products to protect, high-demand products with low promotion need, promotion-sensitive products, and low-performing products that may require pricing or range review.

Modelling grain:
The final modelling dataset will use one row per sold product. Transaction-level, bridge-level, and history-level tables will be aggregated to product level before modelling.
"""

print_tile(size="h3", key="use_case_definition", value=use_case_definition)

---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1: Business-Driven Dataset and Feature Selection

In [ ]:
datasets_used = [
    "sales_order_detail.csv",
    "product.csv",
    "product_sub_category.csv",
    "product_category.csv",
    "special_offer_product.csv",
    "special_offer.csv",
    "product_cost_history.csv",
    "product_list_price_history.csv"
]

datasets_used

In [ ]:
feature_selection_1_insights = """
The first feature selection approach is business-driven dataset selection. The selected use case is product portfolio segmentation for margin-aware promotion decisions, so the preparation focuses on datasets that can describe sold products through demand, price, cost, margin, and promotion behaviour.

The selected datasets are sales_order_detail, product, product_sub_category, product_category, special_offer_product, special_offer, product_cost_history, and product_list_price_history. These datasets support the final modelling grain of one row per sold product.

sales_order_detail is selected as the main transaction fact table because it provides the sales, quantity, price, discount, and actual offer usage behaviour needed to measure product demand and realised commercial performance. product is selected as the main product dimension because it provides product-level price, cost, margin, and descriptive product attributes.

product_sub_category and product_category are selected to provide readable product hierarchy information for interpreting clusters. special_offer_product and special_offer are selected because they provide promotion eligibility, offer type, and discount information. product_cost_history and product_list_price_history are selected only after aggregation because they contain historical cost and price records rather than one row per product.

Datasets such as customer, person, store, sales_territory, and unit_measure are excluded from the first modelling dataset because they do not directly support the product-level margin and promotion clustering objective. Excluding customer and person-level fields also supports data minimisation and avoids unnecessary privacy risk.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2: Demand and Sales Performance Feature Selection

In [ ]:
# Approach 2: Select demand and sales performance features from sales_order_detail

sales_order_detail_feature_selection_df = pd.DataFrame({
    "source_dataset": [
        "sales_order_detail",
        "sales_order_detail",
        "sales_order_detail",
        "sales_order_detail",
        "sales_order_detail",
        "sales_order_detail",
        "sales_order_detail"
    ],
    "source_column": [
        "product_id",
        "sales_order_detail_id",
        "sales_order_id",
        "order_quantity",
        "line_total",
        "unit_price",
        "unit_price_discount"
    ],
    "selected_product_level_feature": [
        "product_id",
        "order_line_count",
        "order_count",
        "total_quantity, avg_order_quantity",
        "total_sales_value, avg_line_value",
        "avg_unit_price",
        "discounted_line_share"
    ],
    "feature_purpose": [
        "Grouping and join key used to aggregate order-line records to one row per product.",
        "Measures how often each product appears in transaction lines.",
        "Measures how many orders contain each product.",
        "Measures total demand and typical quantity purchased per line.",
        "Measures total revenue contribution and average line value.",
        "Measures typical realised selling price in transactions.",
        "Measures how often a product is sold with a realised line-level discount."
    ],
    "reason_for_selection": [
        "Retained for joining and mapping cluster labels back to products, but not used as a clustering input.",
        "High order-line activity can identify frequently purchased products.",
        "Useful candidate feature for demand breadth, although it may overlap with order_line_count.",
        "Quantity features separate high-volume products from low-volume products.",
        "Sales value features separate high-revenue products from low-revenue products.",
        "Unit price helps distinguish low-price products from premium products.",
        "Discount share helps distinguish full-price products from products that rely more on discounting."
    ]
})

sales_order_detail_feature_selection_df

In [ ]:
feature_selection_2_insights = """
The second feature selection approach focuses on demand and sales context using sales_order_detail. This dataset is the main transaction fact table and has one row per sales order line, so its fields must be aggregated to product level before clustering.

product_id is retained as the grouping and join key because it allows order-line records to be aggregated into one row per sold product. However, it is not used as a clustering input because an identifier does not measure product similarity.

The selected behavioural features describe how strongly each product sells and how much revenue it contributes. order_line_count, total_quantity, and avg_order_quantity describe product demand and purchase frequency. order_count is retained as a candidate demand-breadth feature, but it will be checked for overlap with order_line_count before final modelling. total_sales_value and avg_line_value describe revenue contribution and transaction value. avg_unit_price captures the typical realised selling price observed in transactions.

unit_price_discount is used to create discounted_line_share, which measures the proportion of order lines where a product was sold with a discount. This is included because realised discount behaviour is directly relevant to margin-aware promotion decisions.

This feature group supports the business objective because promotion and margin decisions should be interpreted in the context of demand and sales value. For example, a high-demand product with low discount exposure may need discount protection, while a low-demand product with high discount exposure may require pricing or range review.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.3 Approach 3: Price, Cost, and Margin Feature Selection

In [ ]:
# Approach 3: Select price, cost, and margin features from product

price_cost_margin_feature_selection_df = pd.DataFrame({
    "source_dataset": [
        "product",
        "product",
        "product",
        "product",
        "product"
    ],
    "source_column": [
        "product_id",
        "list_price",
        "standard_cost",
        "list_price, standard_cost",
        "list_price, standard_cost"
    ],
    "selected_product_level_feature": [
        "product_id",
        "list_price",
        "standard_cost",
        "gross_margin_amount",
        "gross_margin_pct"
    ],
    "feature_purpose": [
        "Product key used to join product attributes to the sold-product modelling table.",
        "Represents the product's listed selling price before discounts.",
        "Represents the estimated cost of the product to the business.",
        "Measures the absolute difference between list price and standard cost.",
        "Measures the relative margin percentage based on list price and standard cost."
    ],
    "reason_for_selection": [
        "Retained for joining and mapping cluster labels back to products, but not used as a clustering input.",
        "Helps distinguish low-price products from premium products.",
        "Supports cost and profitability analysis at product level.",
        "Shows estimated margin amount before discounts and other costs.",
        "Allows margin comparison across products with different price levels and supports margin-aware promotion decisions."
    ]
})

price_cost_margin_feature_selection_df

In [ ]:
feature_selection_3_insights = """
The third feature selection approach focuses on price, cost, and margin features from product.csv. This feature group is central to the selected use case because the clustering model is intended to support margin-aware promotion decisions, not only identify products with high or low sales.

list_price is selected because it represents the product's listed selling price before discounts. standard_cost is selected because it represents the estimated cost of the product to the business. Together, these features help distinguish low-price products, premium products, and products with different cost structures.

gross_margin_amount and gross_margin_pct are derived from list_price and standard_cost. gross_margin_amount shows the estimated absolute margin available before discounts and other costs, while gross_margin_pct allows profitability comparison across products with different price levels. These features are important because products with high margin may need protection from unnecessary discounting, while products with low margin and high promotion exposure may require pricing or range review.

product_id is retained as the join key and product identifier, but it is not used as a clustering input because an identifier does not measure product similarity.

This feature group supports the business objective by connecting product clusters to margin protection, pricing review, and promotion decision-making.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_3_insights', value=feature_selection_3_insights)

### A.4 Promotion Behaviour Feature Selection


In [ ]:
# Approach 4: Select promotion behaviour features from offer and transaction tables

promotion_feature_selection_df = pd.DataFrame({
    "source_dataset": [
        "special_offer_product, special_offer",
        "special_offer_product, special_offer",
        "sales_order_detail, special_offer",
        "sales_order_detail, special_offer"
    ],
    "source_column": [
        "special_offer_id, product_id, description",
        "discount_pct",
        "special_offer_id, description",
        "discount_pct"
    ],
    "selected_product_level_feature": [
        "promotional_offer_count",
        "max_eligible_discount_pct",
        "promotional_order_line_share",
        "max_actual_discount_pct"
    ],
    "feature_purpose": [
        "Counts how many promotional offer rules a product is eligible or linked to.",
        "Measures the highest discount percentage a product is eligible for.",
        "Measures the proportion of actual sales order lines where a product was sold using a promotional offer.",
        "Measures the highest discount percentage actually used when the product was sold."
    ],
    "reason_for_selection": [
        "Identifies products with greater promotion exposure or offer eligibility.",
        "Helps identify products that could be exposed to deeper discounting.",
        "Separates products that are merely eligible for promotions from products that are actually sold through promotions.",
        "Supports margin-aware review by showing the deepest realised discount behaviour."
    ]
})

promotion_feature_selection_df

In [ ]:
feature_selection_4_insights = """
The fourth feature selection approach focuses on promotion behaviour. This feature group is central to the selected use case because the objective is to support margin-aware promotion decisions across the sold-product portfolio.

Promotion features are selected from two perspectives. special_offer_product joined with special_offer describes promotion eligibility, meaning which products are linked to promotional offer rules and the maximum discount they could receive. sales_order_detail joined with special_offer describes realised promotion usage, meaning which offers were actually used when products were sold.

promotional_offer_count and max_eligible_discount_pct are selected to represent promotion exposure. These features help identify products that are eligible for more promotional treatment or deeper possible discounts. promotional_order_line_share and max_actual_discount_pct are selected to represent realised promotion behaviour in actual transactions.

This distinction is important because a product can be eligible for promotions without being frequently sold through promotions. For margin-aware decision-making, the business needs to know both what discounts are available to a product and what discounts are actually being used.

This feature group supports the business objective by helping identify products that may need discount protection, products that are promotion-sensitive, and products where promotional activity may require review because it could reduce commercial value.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_4_insights', value=feature_selection_4_insights)

### A.5 Approach 5: Historical Price and Cost Movement Feature Selection

In [ ]:
# Approach 5: Select historical price and cost movement features

historical_price_cost_feature_selection_df = pd.DataFrame({
    "source_dataset": [
        "product_list_price_history",
        "product_list_price_history",
        "product_cost_history",
        "product_cost_history"
    ],
    "source_column": [
        "product_id, start_date",
        "list_price",
        "product_id, start_date",
        "standard_cost"
    ],
    "selected_product_level_feature": [
        "price_history_count",
        "list_price_range",
        "cost_history_count",
        "standard_cost_range"
    ],
    "feature_purpose": [
        "Counts how many historical list-price records exist for each product.",
        "Measures how much the product's list price changed across available history.",
        "Counts how many historical standard-cost records exist for each product.",
        "Measures how much the product's standard cost changed across available history."
    ],
    "reason_for_selection": [
        "Products with more price records may have experienced more price updates or lifecycle changes.",
        "Useful for identifying products with stable pricing versus products with larger historical price movement.",
        "Products with more cost records may have experienced more cost updates or lifecycle changes.",
        "Useful for identifying products with stable cost structure versus products with larger historical cost movement."
    ]
})

historical_price_cost_feature_selection_df

In [ ]:
feature_selection_5_insights = """
The fifth feature selection approach focuses on historical price and cost movement. This feature group supports the margin-aware promotion use case because products with unstable prices or costs may need different promotional treatment from products with stable commercial settings.

product_list_price_history and product_cost_history are history tables, so they can contain multiple records per product. They are not joined directly into the final modelling dataset because doing so would duplicate product rows. Instead, they are aggregated to the final modelling grain of one row per sold product.

price_history_count and cost_history_count are selected to indicate how many historical price or cost records exist for each product. list_price_range and standard_cost_range are selected to measure how much the product's listed price or standard cost changed across the available history.

These features provide supporting context for promotion and margin decisions. For example, a product with high promotion exposure and changing cost may require closer margin review than a product with stable cost and low discount usage.

This feature group is used as historical commercial context rather than as raw time-series modelling. It keeps the final clustering dataset simple, interpretable, and aligned with one row per sold product.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_5_insights', value=feature_selection_5_insights)

### Approach 6: Select product hierarchy and profiling features for interpretation

In [ ]:
# Approach 6: Select product hierarchy and profiling features for interpretation

product_hierarchy_profile_feature_selection_df = pd.DataFrame({
    "source_dataset": [
        "product",
        "product",
        "product",
        "product",
        "product",
        "product",
        "product_sub_category, product_category",
        "product_sub_category, product_category"
    ],
    "source_column": [
        "product_id",
        "name",
        "product_number",
        "product_line",
        "class",
        "style",
        "product_subcategory_id",
        "product_category_id"
    ],
    "selected_profile_feature": [
        "product_id",
        "name",
        "product_number",
        "product_line",
        "class",
        "style",
        "subcategory_name",
        "category_name"
    ],
    "modelling_role": [
        "Key / profile column",
        "Profile column",
        "Profile column",
        "Profile column",
        "Profile column",
        "Profile column",
        "Profile column",
        "Profile column"
    ],
    "feature_purpose": [
        "Identifies each product and maps cluster labels back to product records.",
        "Provides a readable product name for interpretation.",
        "Provides a business product code for lookup and reporting.",
        "Provides product-line context where available.",
        "Provides product class context where available.",
        "Provides product style context where available.",
        "Adds readable subcategory information for cluster interpretation.",
        "Adds readable category information for cluster interpretation."
    ],
    "reason_for_selection": [
        "Needed for joining, reporting, and explaining which products belong to each cluster, but not used for distance calculation.",
        "Helps business users interpret cluster membership.",
        "Supports product lookup and reporting.",
        "Useful for profiling clusters after labels are assigned.",
        "Useful for profiling clusters after labels are assigned.",
        "Useful for profiling clusters after labels are assigned.",
        "Helps explain whether clusters are dominated by Road Bikes, Mountain Bikes, Frames, Clothing, or other product groups.",
        "Helps explain whether clusters are dominated by Bikes, Components, Clothing, or Accessories."
    ]
})

product_hierarchy_profile_feature_selection_df

In [ ]:
feature_selection_6_insights = """
The sixth feature selection approach focuses on product hierarchy and profiling features. These fields are retained so that the final clusters can be interpreted and communicated to merchandising stakeholders.

product_id, name, and product_number are retained as profile fields because they allow cluster labels to be mapped back to actual products. They are not used as clustering inputs because identifiers and product names do not measure behavioural similarity.

category_name and subcategory_name are selected from the product hierarchy tables because they provide readable business context. They help explain whether a cluster is dominated by Bikes, Components, Clothing, Accessories, or more specific product subcategories.

product_line, class, and style are retained as additional profiling fields where available. These fields may help describe the types of products inside each cluster after the model has assigned cluster labels. However, they are not treated as core clustering inputs because they are categorical and have missing values.

This feature group supports the business objective by making the clustering output interpretable. The model groups products using numeric commercial behaviour, while the profile fields help explain what those groups mean in product and merchandising terms.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_6_insights', value=feature_selection_6_insights)

### A.7 Approach 7 : "Ethical Feature Selection"

In [ ]:
ethical_exclusions = [
    "person_id",
    "customer_id",
    "store_id",
    "account_number",
    "first_name",
    "middle_name",
    "last_name",
    "title",
    "suffix",
    "additional_contact_info",
    "email_promotion"
]

In [ ]:
feature_selection_7_insights = """
Customer and person-level variables were excluded from the clustering dataset because the selected use case is product segmentation rather than customer segmentation. The model does not require names, account numbers, contact details, or customer identifiers to group products by commercial behaviour.

This reduces privacy risk and supports data minimisation, because only variables needed for the business objective are used. It also avoids creating product clusters that may indirectly profile customer groups or lead to unfair targeting based on personal characteristics. Product identifiers are retained only to map cluster labels back to products after modelling, not as clustering inputs.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_7_insights', value=feature_selection_7_insights)

### A.z Final Selection of Features

In [ ]:
selected_source_columns = [
    # Shared keys
    "product_id",
    "sales_order_id",
    "sales_order_detail_id",
    "special_offer_id",
    "product_subcategory_id",
    "product_category_id",

    # sales_order_detail
    "order_quantity",
    "line_total",
    "unit_price",
    "unit_price_discount",

    # product
    "name",
    "product_number",
    "product_line",
    "class",
    "style",
    "color",
    "list_price",
    "standard_cost",

    # special_offer
    "description",
    "type",
    "category",
    "discount_pct",
    "max_quantity",

    # history tables
    "start_date",
    "end_date"
]

In [ ]:
# Keep only selected columns from the datasets used in this clustering use case

sales_order_detail_df = sales_order_detail_df[
    [
        "sales_order_detail_id",
        "sales_order_id",
        "product_id",
        "order_quantity",
        "line_total",
        "unit_price",
        "unit_price_discount",
        "special_offer_id"
    ]
].copy()

product_df = product_df[
    [
        "product_id",
        "name",
        "product_number",
        "product_subcategory_id",
        "product_line",
        "class",
        "style",
        "color",
        "list_price",
        "standard_cost",
        "sell_end_date"
    ]
].copy()

product_sub_category_df = product_sub_category_df[
    [
        "product_subcategory_id",
        "product_category_id",
        "name"
    ]
].copy()

product_category_df = product_category_df[
    [
        "product_category_id",
        "name"
    ]
].copy()

special_offer_product_df = special_offer_product_df[
    [
        "special_offer_id",
        "product_id"
    ]
].copy()

special_offer_df = special_offer_df[
    [
        "special_offer_id",
        "description",
        "type",
        "category",
        "discount_pct",
        "max_quantity"
    ]
].copy()

product_cost_history_df = product_cost_history_df[
    [
        "product_id",
        "start_date",
        "end_date",
        "standard_cost"
    ]
].copy()

product_list_price_history_df = product_list_price_history_df[
    [
        "product_id",
        "start_date",
        "end_date",
        "list_price"
    ]
].copy()

In [ ]:
feature_selection_explanations = """
The selected source columns were chosen to support the final use case of product portfolio segmentation for margin-aware promotion decisions. The final modelling grain is one row per sold product, so the selected columns include the keys, transaction measures, product commercial attributes, offer fields, product hierarchy fields, and historical price/cost fields needed to build product-level features.

From sales_order_detail, transaction-level columns such as order_quantity, line_total, unit_price, unit_price_discount, sales_order_id, and sales_order_detail_id are selected so they can be aggregated into demand, sales value, price, and discount behaviour features. product_id is retained as the grouping and join key, but it is not used directly as a clustering input.

From product, list_price and standard_cost are selected because they support price, cost, and margin features. Product descriptive fields such as name, product_number, product_line, class, style, and color are retained for cluster profiling and interpretation. product_subcategory_id is selected to join the product hierarchy.

From product_sub_category and product_category, subcategory_name and category_name are selected to make the final clusters understandable in business terms. These hierarchy fields help explain what product types are present in each cluster, but they are not treated as core distance-based clustering inputs.

From special_offer, special_offer_product, and sales_order_detail, offer-related columns such as special_offer_id, discount_pct, description, and type are selected to create promotion eligibility and actual promotion usage features. These are central to identifying products that are promotion-exposed or discount-sensitive.

From product_list_price_history and product_cost_history, start_date, end_date, list_price, and standard_cost are used to derive historical price and cost movement features. The raw history records are not joined directly because they can contain multiple rows per product; they are aggregated before modelling.

Overall, the selected source columns provide the information needed to create final product-level clustering inputs for demand, sales value, price, cost, margin, promotion behaviour, and historical commercial movement, while profile fields are retained separately for interpretation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### Clean sales_order_detail

### B.1 Fixing Clean "sales_order_detail" dataset

In [ ]:
# B.1 Validate sales_order_detail table before feature engineering

sales_order_detail_clean_df = sales_order_detail_df.copy()

sales_order_detail_cleaning_summary_df = pd.DataFrame({
    "cleaning_check": [
        "row_count",
        "unique_sales_order_detail_id",
        "sales_order_detail_id_is_unique",
        "exact_duplicate_rows",
        "missing_value_count",
        "unique_sales_order_id",
        "unique_product_id",
        "unique_special_offer_id"
    ],
    "result": [
        len(sales_order_detail_clean_df),
        sales_order_detail_clean_df["sales_order_detail_id"].nunique(),
        sales_order_detail_clean_df["sales_order_detail_id"].is_unique,
        sales_order_detail_clean_df.duplicated().sum(),
        sales_order_detail_clean_df.isna().sum().sum(),
        sales_order_detail_clean_df["sales_order_id"].nunique(),
        sales_order_detail_clean_df["product_id"].nunique(),
        sales_order_detail_clean_df["special_offer_id"].nunique()
    ]
})

sales_order_detail_cleaning_summary_df

In [ ]:
data_cleaning_1_explanations = """
The sales_order_detail table was reviewed during EDA and confirmed to be a clean order-line fact table. It contains one row per sales order line, with no missing values, no exact duplicate rows, and unique sales_order_detail_id values.

No row removal or value correction is required for this dataset in the data cleaning stage. This is important because sales_order_detail is the main transaction source for product demand, sales value, realised price, discount, and promotion behaviour features.

Extreme values in quantity, unit price, and line total are not removed during cleaning because they may represent genuine business behaviour such as premium products or larger order quantities. Skewed numeric values will be handled later during data preparation for modelling using transformation and scaling, which is more appropriate for K-Means clustering.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "Fixing product Duplicate Records and Unusable Null Fields"

In [ ]:
# B.2 Clean product table

product_clean_df = product_df.copy()

product_rows_before = len(product_clean_df)
product_exact_duplicates = product_clean_df.duplicated().sum()

# Remove exact duplicate product records
product_clean_df = product_clean_df.drop_duplicates().copy()

# Drop discontinued_date because it is fully missing and not useful for modelling
dropped_columns = []

if "discontinued_date" in product_clean_df.columns:
    if product_clean_df["discontinued_date"].isna().all():
        product_clean_df = product_clean_df.drop(columns=["discontinued_date"])
        dropped_columns.append("discontinued_date")

product_cleaning_summary_df = pd.DataFrame({
    "cleaning_check": [
        "rows_before_duplicate_removal",
        "exact_duplicate_rows_removed",
        "rows_after_duplicate_removal",
        "unique_product_id_after_cleaning",
        "product_id_is_unique_after_cleaning",
        "fully_null_columns_dropped",
        "missing_standard_cost_after_cleaning",
        "missing_product_line_after_cleaning",
        "missing_class_after_cleaning",
        "missing_style_after_cleaning",
        "missing_color_after_cleaning"
    ],
    "result": [
        product_rows_before,
        product_exact_duplicates,
        len(product_clean_df),
        product_clean_df["product_id"].nunique(),
        product_clean_df["product_id"].is_unique,
        ", ".join(dropped_columns) if dropped_columns else "None",
        product_clean_df["standard_cost"].isna().sum(),
        product_clean_df["product_line"].isna().sum(),
        product_clean_df["class"].isna().sum(),
        product_clean_df["style"].isna().sum(),
        product_clean_df["color"].isna().sum()
    ]
})

product_cleaning_summary_df

In [ ]:
data_cleaning_2_explanations = """
The second cleaning step prepares product.csv, which is the main product dimension used to add price, cost, margin, and product profile information to the final sold-product modelling dataset.

EDA showed that product.csv contains exact duplicate rows. In preparation, 382 exact duplicate records are removed, reducing the table from 886 rows to 504 rows. After duplicate removal, product_id is unique, which means the cleaned product table can be safely joined to the sold-product modelling table without duplicating product records.

The discontinued_date column was excluded during feature selection because it was fully missing and provided no usable information for clustering or cluster interpretation. Other missing fields are documented but not imputed in this cleaning step. standard_cost has 52 missing values in the cleaned product table, while product_line, class, style, and color also contain missing values.

These missing values are not filled during data cleaning because they require modelling-context decisions. standard_cost will be handled later when price, cost, and margin features are prepared. Product profile fields such as product_line, class, style, and color are retained for interpretation where available, but they are not core clustering inputs. This keeps the cleaning stage focused on clear data quality issues while preserving useful product information for later preparation and profiling.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "Fixing product_list_price_history Duplicate Records"

In [ ]:
# B.3 Clean product_list_price_history table

product_list_price_history_clean_df = product_list_price_history_df.copy()

price_history_rows_before = len(product_list_price_history_clean_df)
price_history_exact_duplicates = product_list_price_history_clean_df.duplicated().sum()

# Remove exact duplicate price-history records
product_list_price_history_clean_df = (
    product_list_price_history_clean_df
    .drop_duplicates()
    .copy()
)

# Convert date fields to datetime for later history aggregation
product_list_price_history_clean_df["start_date"] = pd.to_datetime(
    product_list_price_history_clean_df["start_date"],
    errors="coerce"
)

product_list_price_history_clean_df["end_date"] = pd.to_datetime(
    product_list_price_history_clean_df["end_date"],
    errors="coerce"
)

price_history_cleaning_summary_df = pd.DataFrame({
    "cleaning_check": [
        "rows_before_duplicate_removal",
        "exact_duplicate_rows_removed",
        "rows_after_duplicate_removal",
        "unique_products_after_cleaning",
        "duplicate_product_id_start_date_after_cleaning",
        "missing_start_date_after_cleaning",
        "missing_end_date_after_cleaning",
        "missing_list_price_after_cleaning"
    ],
    "result": [
        price_history_rows_before,
        price_history_exact_duplicates,
        len(product_list_price_history_clean_df),
        product_list_price_history_clean_df["product_id"].nunique(),
        product_list_price_history_clean_df.duplicated(["product_id", "start_date"]).sum(),
        product_list_price_history_clean_df["start_date"].isna().sum(),
        product_list_price_history_clean_df["end_date"].isna().sum(),
        product_list_price_history_clean_df["list_price"].isna().sum()
    ]
})

price_history_cleaning_summary_df

In [ ]:
data_cleaning_3_explanations = """
The third cleaning step prepares product_list_price_history, which is used later to create historical price movement features.

The raw table contains 620 rows and 225 exact duplicate records. These duplicates are removed, leaving 395 rows across 293 unique products. After duplicate removal, there are no duplicate product_id-start_date records, which means the cleaned table is suitable for later product-level aggregation.

The start_date and end_date fields are converted to datetime so they can be used safely in later history-based feature engineering. Missing end_date values are not treated as data errors because they may represent open or current price records. Missing start_date and list_price values are documented but not imputed in the cleaning stage.

This table is not joined directly into the clustering dataset because it can contain multiple records per product. Instead, the cleaned table will later be aggregated into product-level historical price movement features, such as list_price_range, while preserving the final modelling grain of one row per sold product.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.4 Cleaning product_cost_history Date Fields and Open Records"

In [ ]:
# B.4 Clean product_cost_history table

product_cost_history_clean_df = product_cost_history_df.copy()

cost_history_rows_before = len(product_cost_history_clean_df)
cost_history_exact_duplicates = product_cost_history_clean_df.duplicated().sum()

# Remove exact duplicate rows if any are present
product_cost_history_clean_df = product_cost_history_clean_df.drop_duplicates().copy()

# Convert date fields to datetime for later history aggregation
product_cost_history_clean_df["start_date"] = pd.to_datetime(
    product_cost_history_clean_df["start_date"],
    errors="coerce"
)

product_cost_history_clean_df["end_date"] = pd.to_datetime(
    product_cost_history_clean_df["end_date"],
    errors="coerce"
)

# Missing end_date represents an open/current cost record, not a data error
product_cost_history_clean_df["is_open_cost_record"] = (
    product_cost_history_clean_df["end_date"].isna()
)

cost_history_cleaning_summary_df = pd.DataFrame({
    "cleaning_check": [
        "rows_before_duplicate_removal",
        "exact_duplicate_rows_removed",
        "rows_after_duplicate_removal",
        "unique_products_after_cleaning",
        "duplicate_product_id_start_date_after_cleaning",
        "missing_start_date_after_cleaning",
        "missing_end_date_after_cleaning",
        "missing_standard_cost_after_cleaning",
        "open_cost_records"
    ],
    "result": [
        cost_history_rows_before,
        cost_history_exact_duplicates,
        len(product_cost_history_clean_df),
        product_cost_history_clean_df["product_id"].nunique(),
        product_cost_history_clean_df.duplicated(["product_id", "start_date"]).sum(),
        product_cost_history_clean_df["start_date"].isna().sum(),
        product_cost_history_clean_df["end_date"].isna().sum(),
        product_cost_history_clean_df["standard_cost"].isna().sum(),
        product_cost_history_clean_df["is_open_cost_record"].sum()
    ]
})

cost_history_cleaning_summary_df

In [ ]:
data_cleaning_4_explanations = """
The fourth cleaning step prepares product_cost_history, which is used later to create historical cost movement features and support product cost analysis.

The table contains 395 rows, with no exact duplicate records and no duplicate product_id-start_date records. This confirms that the history table is clean at its natural grain, where each row represents a product cost record for a specific start period.

The start_date and end_date fields are converted to datetime so the table can be used safely in later history-based feature engineering. There are no missing start_date or standard_cost values. The 195 missing end_date values are not treated as data errors because they represent open or current cost records. An is_open_cost_record flag is created to make this interpretation explicit for later preparation work.

The raw cost history table is not joined directly into the final clustering dataset because it can contain multiple records per product. It will later be aggregated into product-level cost movement features, such as standard_cost_range, while preserving the final modelling grain of one row per sold product.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_4_explanations', value=data_cleaning_4_explanations)

### B.5 Validating Offer and Product Hierarchy Reference Tables

In [ ]:
# B.5 Validate offer and product hierarchy reference tables

reference_table_cleaning_summary_df = pd.DataFrame({
    "table": [
        "special_offer",
        "special_offer_product",
        "product_sub_category",
        "product_category"
    ],
    "row_count": [
        len(special_offer_df),
        len(special_offer_product_df),
        len(product_sub_category_df),
        len(product_category_df)
    ],
    "missing_value_count": [
        special_offer_df.isna().sum().sum(),
        special_offer_product_df.isna().sum().sum(),
        product_sub_category_df.isna().sum().sum(),
        product_category_df.isna().sum().sum()
    ],
    "exact_duplicate_rows": [
        special_offer_df.duplicated().sum(),
        special_offer_product_df.duplicated().sum(),
        product_sub_category_df.duplicated().sum(),
        product_category_df.duplicated().sum()
    ],
    "key_or_relationship_duplicates": [
        special_offer_df["special_offer_id"].duplicated().sum(),
        special_offer_product_df.duplicated(["special_offer_id", "product_id"]).sum(),
        product_sub_category_df["product_subcategory_id"].duplicated().sum(),
        product_category_df["product_category_id"].duplicated().sum()
    ]
})

reference_table_cleaning_summary_df

In [ ]:
data_cleaning_5_explanations = """
The fifth cleaning step validates the offer and product hierarchy reference tables used for promotion features and cluster interpretation.

special_offer, special_offer_product, product_sub_category, and product_category are checked for missing values, exact duplicate rows, and duplicate key or relationship records. No duplicate key issues are found: special_offer_id is unique in special_offer, offer-product pairs are unique in special_offer_product, product_subcategory_id is unique in product_sub_category, and product_category_id is unique in product_category.

The only missing values identified in these reference tables are 12 missing max_quantity values in special_offer. These are retained because they represent open-ended offer rules rather than invalid records. max_quantity is not used directly as a clustering input, so no imputation or row removal is required.

Raw categorical fields such as offer description, offer type, category name, and subcategory name are not used directly as numeric clustering inputs in the first model. They are used to derive promotion features or retained for cluster profiling and interpretation. Therefore, no row-level cleaning is required for these reference and bridge tables.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_5_explanations', value=data_cleaning_5_explanations)

### B.6 Repairing Product Missing Values for Profiling and Cost Coverage

In [ ]:
# B.6 Repair product missing values for profiling and cost coverage

# Fill missing product profile fields with "Unknown"
product_profile_text_columns = [
    "name",
    "product_number",
    "product_line",
    "class",
    "style",
    "color"
]

for col in product_profile_text_columns:
    if col in product_clean_df.columns:
        product_clean_df[col] = product_clean_df[col].fillna("Unknown")

# Use latest available standard cost from cost history where product standard_cost is missing
latest_cost_from_history_df = (
    product_cost_history_clean_df
    .sort_values(["product_id", "start_date"])
    .groupby("product_id", as_index=False)
    .tail(1)[["product_id", "standard_cost"]]
    .rename(columns={"standard_cost": "latest_history_standard_cost"})
)

product_clean_df = product_clean_df.merge(
    latest_cost_from_history_df,
    on="product_id",
    how="left",
    validate="one_to_one"
)

product_clean_df["standard_cost_source"] = np.where(
    product_clean_df["standard_cost"].notna(),
    "product_table",
    np.where(
        product_clean_df["latest_history_standard_cost"].notna(),
        "cost_history",
        "missing"
    )
)

product_clean_df["standard_cost"] = product_clean_df["standard_cost"].fillna(
    product_clean_df["latest_history_standard_cost"]
)

product_missing_repair_summary_df = pd.DataFrame({
    "cleaning_check": [
        "standard_cost_from_product_table",
        "standard_cost_repaired_from_cost_history",
        "standard_cost_still_missing",
        "missing_name_after_repair",
        "missing_product_number_after_repair",
        "missing_product_line_after_repair",
        "missing_class_after_repair",
        "missing_style_after_repair",
        "missing_color_after_repair"
    ],
    "result": [
        (product_clean_df["standard_cost_source"] == "product_table").sum(),
        (product_clean_df["standard_cost_source"] == "cost_history").sum(),
        product_clean_df["standard_cost"].isna().sum(),
        product_clean_df["name"].isna().sum(),
        product_clean_df["product_number"].isna().sum(),
        product_clean_df["product_line"].isna().sum(),
        product_clean_df["class"].isna().sum(),
        product_clean_df["style"].isna().sum(),
        product_clean_df["color"].isna().sum()
    ]
})

product_missing_repair_summary_df

In [ ]:
data_cleaning_6_explanations = """
The sixth cleaning step repairs product missing values that affect cost coverage and later cluster interpretation.

Missing product profile fields such as name, product_number, product_line, class, style, and color are filled with "Unknown". These fields are retained for cluster interpretation rather than direct distance-based clustering, so using "Unknown" preserves the product records without forcing an artificial category.

standard_cost is important because it is required for margin-based features. In the cleaned product table, 452 products already had standard_cost available. For products where standard_cost was missing, the latest available standard_cost from product_cost_history was used as a source-based repair. This repaired 30 additional product records, leaving 22 products still missing standard_cost.

A standard_cost_source field is retained to make the repair auditable. It records whether the cost came from product.csv, product_cost_history, or remains missing. Any remaining missing cost values will be handled later during final modelling preparation if they appear in the sold-product modelling dataset.

This cleaning step improves the reliability of later margin features and supports the margin-aware promotion clustering use case.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_6_explanations', value=data_cleaning_6_explanations)

---
## C. Split Datasets


In [ ]:
training_df = pd.DataFrame()
validation_df = pd.DataFrame()
testing_df = pd.DataFrame()

In [ ]:
data_splitting_explanations = """
A supervised train-validation-test split is not required at this stage because clustering is an unsupervised learning task with no target variable. The objective is to discover natural product groups, not predict a known label.

The final product-level clustering dataset has not been created yet because the selected source tables still need to be aggregated and integrated to one row per sold product. Therefore, placeholder DataFrames are created in this section to keep the notebook structure valid, while the actual modelling dataset is built in the Feature Engineering and Data Preparation sections.

After the final model input dataset is prepared, it will be assigned to X_train for export because it represents the complete clustering input matrix. X_val and X_test will remain empty placeholders because supervised validation and test sets are not appropriate for this unsupervised clustering preparation notebook.

Clustering solutions will be evaluated later using internal validation and business interpretability, including inertia, silhouette score, cluster size balance, and the usefulness of cluster profiles for margin-aware promotion decisions.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 Aggregate Transaction Behaviour to Product Level



In [ ]:
# D.1 Aggregate sales_order_detail to product level

sales_order_detail_clean_df = sales_order_detail_df.copy()
sales_order_detail_clean_df["is_discounted"] = (
    sales_order_detail_clean_df["unit_price_discount"] > 0
).astype(int)

product_sales_summary_df = (
    sales_order_detail_clean_df
    .groupby("product_id", as_index=False)
    .agg(
        order_line_count=("sales_order_detail_id", "nunique"),
        order_count=("sales_order_id", "nunique"),
        total_quantity=("order_quantity", "sum"),
        avg_order_quantity=("order_quantity", "mean"),
        total_sales_value=("line_total", "sum"),
        avg_line_value=("line_total", "mean"),
        avg_unit_price=("unit_price", "mean"),
        discounted_line_share=("is_discounted", "mean")
    )
)

print("Product-level sales summary rows:", len(product_sales_summary_df))
print("Unique product_id:", product_sales_summary_df["product_id"].nunique())
print("product_id is unique:", product_sales_summary_df["product_id"].is_unique)

product_sales_summary_df.head()

In [ ]:
feature_engineering_1_explanations = """
The first feature engineering step aggregates sales_order_detail from order-line level to product level. This is required because the final clustering dataset must contain one row per sold product.

The aggregation creates demand and sales context features, including order_line_count, order_count, total_quantity, avg_order_quantity, total_sales_value, avg_line_value, avg_unit_price, and discounted_line_share. These features describe how strongly each product sells, how much revenue it contributes, its realised transaction price, and how often it is sold with a discount.

A binary is_discounted indicator is created before aggregation so discount behaviour can be summarised as discounted_line_share at product level. This is relevant to the margin-aware promotion use case because products with high discount exposure may need promotion review or margin protection.

The output is checked to confirm that product_id is unique, ensuring the result is suitable for joining into the final one-row-per-sold-product modelling dataset.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New features "product price, cost, margin, and profile features"



In [ ]:
# D.2 Create product price, cost, margin, and profile features

product_feature_df = product_clean_df.copy()

# Rename product name to avoid confusion with category/offer names later
product_feature_df = product_feature_df.rename(columns={"name": "product_name"})

# Create gross margin features safely
product_feature_df["gross_margin_amount"] = (
    product_feature_df["list_price"] - product_feature_df["standard_cost"]
)

product_feature_df["gross_margin_pct"] = np.where(
    (product_feature_df["list_price"] > 0) & product_feature_df["standard_cost"].notna(),
    (product_feature_df["gross_margin_amount"] / product_feature_df["list_price"]) * 100,
    np.nan
)

product_feature_df = product_feature_df[
    [
        "product_id",
        "product_name",
        "product_number",
        "product_subcategory_id",
        "product_line",
        "class",
        "style",
        "color",
        "list_price",
        "standard_cost",
        "gross_margin_amount",
        "gross_margin_pct"
    ]
].copy()

print("Product feature rows:", len(product_feature_df))
print("Unique product_id:", product_feature_df["product_id"].nunique())
print("product_id is unique:", product_feature_df["product_id"].is_unique)

product_feature_df[
    ["list_price", "standard_cost", "gross_margin_amount", "gross_margin_pct"]
].isna().sum()

In [ ]:
feature_engineering_2_explanations = """
The second feature engineering step creates product-level price, cost, margin, and profile features from the cleaned product table.

product_name is renamed from the original name column to avoid confusion with category or offer name fields during later joins. list_price and standard_cost are retained because they represent the product's listed selling price and estimated cost to the business.

gross_margin_amount and gross_margin_pct are derived from list_price and standard_cost. These features are central to the margin-aware promotion use case because they help identify products where discounting may reduce commercial value. gross_margin_pct is calculated only when list_price is greater than zero and standard_cost is available, which avoids invalid division by zero.

After product cleaning and cost repair, standard_cost is missing for 22 products, and those missing values flow into gross_margin_amount. gross_margin_pct has more missing values because it also cannot be calculated when list_price is zero. These remaining missing values are documented here and will be handled later during final modelling preparation.

The output remains one row per product and includes both modelling features and profile fields that will later help interpret clusters.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 Create product-level promotion eligibility and actual usage features

In [ ]:
# D.3 Create product-level promotion eligibility and actual usage features

# Promotion eligibility from special_offer_product + special_offer
offer_product_enriched_df = (
    special_offer_product_df
    .merge(
        special_offer_df[
            ["special_offer_id", "description", "type", "category", "discount_pct"]
        ],
        on="special_offer_id",
        how="left",
        validate="many_to_one"
    )
)

offer_product_enriched_df["is_promotional_offer"] = (
    offer_product_enriched_df["description"] != "No Discount"
).astype(int)

product_offer_eligibility_df = (
    offer_product_enriched_df
    .groupby("product_id", as_index=False)
    .agg(
        promotional_offer_count=("is_promotional_offer", "sum"),
        max_eligible_discount_pct=("discount_pct", "max")
    )
)

# Actual promotion usage from sales_order_detail + special_offer
sales_offer_enriched_df = (
    sales_order_detail_clean_df
    .merge(
        special_offer_df[
            ["special_offer_id", "description", "type", "category", "discount_pct"]
        ],
        on="special_offer_id",
        how="left",
        validate="many_to_one"
    )
)

sales_offer_enriched_df["used_promotional_offer"] = (
    sales_offer_enriched_df["description"] != "No Discount"
).astype(int)

product_offer_usage_df = (
    sales_offer_enriched_df
    .groupby("product_id", as_index=False)
    .agg(
        promotional_order_line_share=("used_promotional_offer", "mean"),
        max_actual_discount_pct=("discount_pct", "max")
    )
)

sold_products_df = product_sales_summary_df[["product_id"]].copy()

product_promotion_features_df = (
    sold_products_df
    .merge(product_offer_eligibility_df, on="product_id", how="left")
    .merge(product_offer_usage_df, on="product_id", how="left")
)

promotion_feature_columns = [
    "promotional_offer_count",
    "max_eligible_discount_pct",
    "promotional_order_line_share",
    "max_actual_discount_pct"
]

product_promotion_features_df[promotion_feature_columns] = (
    product_promotion_features_df[promotion_feature_columns].fillna(0)
)

print("Promotion feature rows:", len(product_promotion_features_df))
print("Unique product_id:", product_promotion_features_df["product_id"].nunique())
print("product_id is unique:", product_promotion_features_df["product_id"].is_unique)

product_promotion_features_df.head()

In [ ]:
feature_engineering_3_explanations = """
The third feature engineering step creates product-level promotion features from both promotion eligibility and actual promotion usage.

Promotion eligibility is created by joining special_offer_product with special_offer. This describes which products are linked to promotional offer rules and the maximum discount percentage they are eligible for. Actual promotion usage is created by joining sales_order_detail with special_offer, which describes whether sold products were actually sold through promotional offers.

The final promotion feature table is restricted to the sold-product population from product_sales_summary_df. This is important because the selected clustering use case focuses on products with observed sales behaviour, and the final modelling dataset must remain at one row per sold product.

Missing promotion feature values are filled with zero because no matched eligibility or usage record means no captured promotional exposure or realised promotional usage for that sold product.

These features support the margin-aware promotion use case by distinguishing products that are promotion-exposed, promotion-used, or mainly sold without promotional behaviour.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.4 Create Historical Price and Cost Movement Features

In [ ]:
# D.4 Create historical price and cost movement features

# Historical list price movement
available_price_history_df = product_list_price_history_clean_df[
    product_list_price_history_clean_df["list_price"].notna()
].copy()

product_price_history_summary_df = (
    available_price_history_df
    .groupby("product_id", as_index=False)
    .agg(
        price_history_count=("list_price", "count"),
        min_historical_list_price=("list_price", "min"),
        max_historical_list_price=("list_price", "max")
    )
)

product_price_history_summary_df["list_price_range"] = (
    product_price_history_summary_df["max_historical_list_price"]
    - product_price_history_summary_df["min_historical_list_price"]
)

# Historical standard cost movement
product_cost_history_summary_df = (
    product_cost_history_clean_df
    .groupby("product_id", as_index=False)
    .agg(
        cost_history_count=("standard_cost", "count"),
        min_historical_standard_cost=("standard_cost", "min"),
        max_historical_standard_cost=("standard_cost", "max")
    )
)

product_cost_history_summary_df["standard_cost_range"] = (
    product_cost_history_summary_df["max_historical_standard_cost"]
    - product_cost_history_summary_df["min_historical_standard_cost"]
)

product_history_features_df = (
    product_price_history_summary_df[
        ["product_id", "price_history_count", "list_price_range"]
    ]
    .merge(
        product_cost_history_summary_df[
            ["product_id", "cost_history_count", "standard_cost_range"]
        ],
        on="product_id",
        how="outer",
        validate="one_to_one"
    )
)

print("History feature rows:", len(product_history_features_df))
print("Unique product_id:", product_history_features_df["product_id"].nunique())
print("product_id is unique:", product_history_features_df["product_id"].is_unique)

product_history_features_df.head()

In [ ]:
feature_engineering_4_explanations = """
The fourth feature engineering step creates historical price and cost movement features at product level.

product_list_price_history and product_cost_history are history tables, so they can contain multiple records for the same product. They are not joined directly into the final modelling dataset because doing so would duplicate product rows and break the required modelling grain.

For product_list_price_history, records with missing list_price are excluded from the price aggregation because they cannot contribute to historical price movement. price_history_count and list_price_range are then calculated for each product. list_price_range measures the difference between the highest and lowest available historical list price.

For product_cost_history, cost_history_count and standard_cost_range are calculated for each product. standard_cost_range measures the difference between the highest and lowest historical standard cost.

The resulting history feature table contains one row per product with available price or cost history. It is not restricted to sold products at this step; instead, it will be joined to the sold-product modelling table during final integration. These features provide supporting commercial stability context for the margin-aware promotion use case because products with larger price or cost movement may require closer review before promotion decisions are made.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_4_explanations', value=feature_engineering_4_explanations)

### D.5 Add Product Hierarchy for Cluster Interpretation

In [ ]:
# D.5 Create product hierarchy features for cluster interpretation

# Rename source name columns before merging to avoid ambiguous name_x/name_y columns
product_sub_category_named_df = product_sub_category_df.rename(
    columns={"name": "subcategory_name"}
).copy()

product_category_named_df = product_category_df.rename(
    columns={"name": "category_name"}
).copy()

product_hierarchy_df = (
    product_sub_category_named_df
    .merge(
        product_category_named_df,
        on="product_category_id",
        how="left",
        validate="many_to_one"
    )
)

product_hierarchy_df = product_hierarchy_df[
    [
        "product_subcategory_id",
        "product_category_id",
        "subcategory_name",
        "category_name"
    ]
].copy()

print("Product hierarchy rows:", len(product_hierarchy_df))
print("Unique product_subcategory_id:", product_hierarchy_df["product_subcategory_id"].nunique())
print("product_subcategory_id is unique:", product_hierarchy_df["product_subcategory_id"].is_unique)
print("Missing category_name:", product_hierarchy_df["category_name"].isna().sum())
print("Missing subcategory_name:", product_hierarchy_df["subcategory_name"].isna().sum())

product_hierarchy_df.head()

In [ ]:
feature_engineering_5_explanations = """
The fifth feature engineering step creates product hierarchy fields for cluster interpretation.

product_sub_category is joined with product_category so that each product subcategory can be linked to its broader product category. This creates readable subcategory_name and category_name fields.

The resulting hierarchy table contains 37 rows, with 37 unique product_subcategory_id values and no missing category or subcategory labels. This confirms that the hierarchy table can be safely joined to product records by product_subcategory_id.

These hierarchy fields are not intended to be the main numeric clustering inputs in the first model. Instead, they are retained as profile fields so that cluster results can be interpreted in business terms. For example, after the model assigns cluster labels, category_name and subcategory_name can show whether a cluster is dominated by Bikes, Components, Clothing, or Accessories.

This supports the business use case by making the final product clusters understandable and actionable for merchandising stakeholders.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_5_explanations', value=feature_engineering_5_explanations)

### D.6 Integrate Final Product-Level Modelling Dataset

In [ ]:
# D.6 Integrate final product-level modelling dataset

modelling_df = (
    product_sales_summary_df
    .merge(
        product_feature_df,
        on="product_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        product_hierarchy_df,
        on="product_subcategory_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        product_promotion_features_df,
        on="product_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        product_history_features_df,
        on="product_id",
        how="left",
        validate="one_to_one"
    )
)

print("Final modelling dataset rows:", len(modelling_df))
print("Unique product_id:", modelling_df["product_id"].nunique())
print("product_id is unique:", modelling_df["product_id"].is_unique)

print("\nMissing values in selected modelling/profile columns:")
display(
    modelling_df[
        [
            "product_id",
            "product_name",
            "category_name",
            "subcategory_name",
            "order_line_count",
            "total_quantity",
            "total_sales_value",
            "avg_line_value",
            "avg_unit_price",
            "list_price",
            "standard_cost",
            "gross_margin_amount",
            "gross_margin_pct",
            "discounted_line_share",
            "promotional_offer_count",
            "max_eligible_discount_pct",
            "promotional_order_line_share",
            "max_actual_discount_pct",
            "price_history_count",
            "list_price_range",
            "cost_history_count",
            "standard_cost_range"
        ]
    ].isna().sum()
)

modelling_df.head()

In [ ]:
feature_engineering_6_explanations = """
The sixth feature engineering step integrates the product-level feature tables into one final modelling dataset.

The integration starts from product_sales_summary_df because the clustering population is defined as sold products with observed transaction behaviour. Product price, cost, margin, profile fields, product hierarchy fields, promotion features, and historical price/cost movement features are then joined to this sold-product base.

Each joined table has already been aggregated or cleaned to the correct level before integration. This prevents product duplication and protects the final modelling grain of one row per sold product. Join validation is used to confirm that the joins do not unexpectedly create many-to-many relationships.

The final integrated dataset contains 254 rows and 254 unique product_id values, confirming that the final modelling grain is one row per sold product. The main demand, sales, price, cost, margin, promotion, and hierarchy features have complete coverage after integration.

Only price_history_count and list_price_range contain missing values for 33 sold products. These missing values indicate that no usable historical list price record was available for those products after cleaning. They will be handled in the later data preparation stage before modelling.

This integration step connects the full business story: demand and sales context, margin position, promotion behaviour, historical commercial movement, and readable product hierarchy for interpretation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_6_explanations', value=feature_engineering_6_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Handle remaining missing values in the final modelling dataset


In [ ]:
# E.1 Handle remaining missing values in the final modelling dataset

prepared_modelling_df = modelling_df.copy()

# Missing history values mean no usable historical movement was available
history_missing_fill_columns = [
    "price_history_count",
    "list_price_range",
    "cost_history_count",
    "standard_cost_range"
]

prepared_modelling_df[history_missing_fill_columns] = (
    prepared_modelling_df[history_missing_fill_columns].fillna(0)
)

clustering_features = [
    "order_line_count",
    "total_quantity",
    "total_sales_value",
    "avg_line_value",
    "avg_unit_price",
    "list_price",
    "standard_cost",
    "gross_margin_amount",
    "gross_margin_pct",
    "discounted_line_share",
    "promotional_offer_count",
    "max_eligible_discount_pct",
    "promotional_order_line_share",
    "max_actual_discount_pct",
    "price_history_count",
    "list_price_range",
    "cost_history_count",
    "standard_cost_range"
]

missing_after_fill_df = pd.DataFrame({
    "feature": clustering_features,
    "missing_count": prepared_modelling_df[clustering_features].isna().sum().values
})

missing_after_fill_df

In [ ]:
data_transformation_1_explanations = """
The first data preparation step handles remaining missing values in the final product-level modelling dataset.

After integration, missing values can occur in history-derived fields when a sold product has no usable historical price or cost record after cleaning. In the current dataset, price_history_count and list_price_range required this treatment, while cost_history_count and standard_cost_range already had complete coverage.

All history-derived missing values are filled with 0 because no available history means no observed historical movement in the prepared data. This treatment is applied consistently across price and cost history features to make the preparation logic robust and reproducible.

After this treatment, all selected clustering features have zero missing values. This confirms that the final model input can be prepared without unresolved null values.

This step is necessary because clustering algorithms such as K-Means cannot handle missing values directly, and unresolved missing values would prevent model training or distort distance calculations.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Log transform skewed numeric features

In [ ]:
# E.2 Log transform skewed numeric features

model_transform_df = prepared_modelling_df.copy()

skewed_features_to_log = [
    "order_line_count",
    "total_quantity",
    "total_sales_value",
    "avg_line_value",
    "avg_unit_price",
    "list_price",
    "standard_cost",
    "gross_margin_amount",
    "promotional_offer_count",
    "price_history_count",
    "list_price_range",
    "cost_history_count",
    "standard_cost_range"
]

for feature in skewed_features_to_log:
    model_transform_df[feature] = np.log1p(model_transform_df[feature])

log_transformation_summary_df = pd.DataFrame({
    "feature": skewed_features_to_log,
    "original_skew": prepared_modelling_df[skewed_features_to_log].skew().values,
    "skew_after_log1p": model_transform_df[skewed_features_to_log].skew().values
})

log_transformation_summary_df

In [ ]:
data_transformation_2_explanations = """
The second data preparation step applies a log1p transformation to non-negative numeric features that are unbounded or count-like.

Several candidate clustering features are right-skewed because a small number of products have much higher sales activity, sales value, realised price, listed price, cost, dollar margin, promotion exposure, or historical price/cost movement than most products. These extreme values are valid business observations, not data errors, because premium, high-demand, and high-revenue products are important for product portfolio segmentation.

The log1p transformation compresses large values more than small values while preserving the relative ordering of products. This reduces the influence of extreme values before distance-based clustering.

The transformation was effective for the main sales, demand, price, cost, and margin amount features. For example, order_line_count reduced from 3.70 to 0.03 skewness, total_quantity from 3.21 to -0.36, total_sales_value from 2.85 to -0.12, avg_unit_price from 2.03 to -0.16, list_price from 1.61 to -0.40, standard_cost from 1.56 to -0.38, and gross_margin_amount from 1.73 to -0.40.

The historical movement features improved but remained moderately right-skewed. This is expected because many products have little or no historical price or cost movement, while a smaller number of products have larger changes. These features are retained because they provide useful commercial stability context.

Bounded percentage and share features, such as gross_margin_pct, discounted_line_share, max_eligible_discount_pct, promotional_order_line_share, and max_actual_discount_pct, are not log-transformed because they are already measured on controlled ratio or percentage scales.

This improves the modelling dataset because the clustering algorithm is less likely to be dominated by extreme product values, while still keeping commercially important products in the analysis.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Scale clustering features


In [ ]:
# E.3 Scale clustering features

scaling_features = clustering_features.copy()

scaler = StandardScaler()

scaled_feature_array = scaler.fit_transform(model_transform_df[scaling_features])

scaled_feature_df = pd.DataFrame(
    scaled_feature_array,
    columns=scaling_features,
    index=model_transform_df.index
)

scaling_summary_df = pd.DataFrame({
    "feature": scaling_features,
    "scaled_mean": scaled_feature_df.mean().round(6).values,
    "scaled_std": scaled_feature_df.std(ddof=0).round(6).values
})

scaling_summary_df

In [ ]:
data_transformation_3_explanations = """
The third transformation standardises the numeric clustering features using StandardScaler.

This is important because the selected features are measured on very different scales. For example, total_sales_value and list_price can be much larger numeric values, while discount shares, discount percentages, and margin percentages are measured on smaller scales. Without scaling, distance-based clustering algorithms could give too much influence to large-value features simply because of their units.

StandardScaler transforms each feature so that it has a mean close to 0 and a standard deviation close to 1. The scaling summary confirms this, with all transformed features showing approximately 0 mean and 1 standard deviation.

This improves the modelling dataset because demand, revenue, price, margin, promotion, and historical movement features can contribute more fairly to the clustering distance calculation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.4 Create final modelling inputs

In [ ]:
# E.4 Create final modelling inputs

profile_columns = [
    "product_id",
    "product_name",
    "product_number",
    "category_name",
    "subcategory_name",
    "product_line",
    "class",
    "style",
    "color"
]

final_modelling_profile_df = prepared_modelling_df[profile_columns].copy()

final_model_input_df = pd.concat(
    [
        final_modelling_profile_df.reset_index(drop=True),
        scaled_feature_df.reset_index(drop=True)
    ],
    axis=1
)

X_train = final_model_input_df.copy()
X_val = pd.DataFrame()
X_test = pd.DataFrame()

print("Final model input rows:", len(final_model_input_df))
print("Final model input columns:", final_model_input_df.shape[1])
print("Unique product_id:", final_model_input_df["product_id"].nunique())
print("product_id is unique:", final_model_input_df["product_id"].is_unique)

final_model_input_df.head()

In [ ]:
data_transformation_4_explanations = """
The final preparation step creates the modelling input dataset for the clustering notebooks.

The dataset keeps product profile columns such as product_id, product name, category, subcategory, product line, class, style, and colour so that cluster labels can later be mapped back to meaningful business information. These profile columns are not intended to be used directly as numeric clustering inputs.

The scaled numeric features are then added to the profile columns. These transformed features represent demand, sales value, price, margin, promotion behaviour, and historical movement on a comparable scale.

Because this is an unsupervised clustering problem, there is no target variable and no conventional train-validation-test split. Therefore, the full prepared product-level dataset is assigned to X_train, while X_val and X_test are kept as empty placeholder DataFrames for notebook structure.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_4_explanations', value=data_transformation_4_explanations)

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)